In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from bayes_opt import BayesianOptimization
import shap
import os
from pathlib import Path
import joblib

# 【关键修改】确保 SVG 中的文字在 Figma 中可编辑，而不是变成路径
plt.rcParams['svg.fonttype'] = 'none' 
# 设置全局字体为 Times New Roman
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12

In [3]:
# ================== 路径与数据加载 ==================
file_path = r"C:\Users\GIGABYTE\Desktop\Test\Analysis\新（可用）\CSV\0312_2000_North0°_Overhang+Vertical(1)_replaced.csv"
data = pd.read_csv(file_path)

# 定义需要删除的无关列
columns_to_drop = ['in:Index', 'img:Output1', 'img:Output2', 'img:Output3', 'img:Output4', 
                  'out:Degree', 'out:Rank', 'out:Hypervolume', 'out:WWR_Skylight', 'out:W_Skylight']
data = data.drop(columns=[col for col in columns_to_drop if col in data.columns])

# =============================================================================
# 第一部分：安全列名映射表 (Safe Rename Dict)
# 用途：用于模型训练 (XGBoost, LightGBM, Random Forest)
# 特点：只包含字母、数字和下划线，避免任何 LaTeX 符号或空格，确保算法库不报错。
# =============================================================================
safe_rename_dict = {
    # --- 外窗侧墙 (Skylight side) ---
    "WWR_Skylight": "WWR_s",
    "WWR_Skylight_Vertical_Shading_Divided": "WWR_s", # 垂直遮阳特殊处理
    "H_Skylight": "H_s",
    "W_Skylight": "W_s",
    "W_Skylight_Vertical_Shading_Divided": "W_s",     # 垂直遮阳特殊处理
    "SH_Skylight": "SH_s",
    "Distance_A_Skylight": "d_A_s",
    "Distance_B_Skylight": "d_B_s",
    
    # --- 走廊侧墙 (Corridor side) ---
    "WWR_Corridor": "WWR_c",
    "H_Corridor": "H_c",
    "W_Corridor": "W_c",
    "SH_Corridor": "SH_c",
    "Distance_A_Corridor": "d_A_c",
    "Distance_B_Corridor": "d_B_c",
    
    # --- 框式集成遮阳 (Frame) ---
    "Frame_BL": "F_BL",
    "Frame_BR": "F_BR",
    "Frame_TL": "F_TL",
    "Frame_TR": "F_TR",
    
    # --- 悬挑遮阳 (Overhang) ---
    "Overhang_Angle": "alpha_oh",
    "Overhang_Length": "L_oh",
    "Overhang_Movement_Distance": "d_mv",
    
    # --- 垂直遮阳 (Vertical Shading) ---
    "Vertical_Shading_Number": "N_v",
    "Vertical_Shading_Width": "W_v",
    "Vertical_Shading_Length": "L_v",
    # 注意：此处包含了您 CSV 表头中的原始拼写错误 "Vertiacl" 以确保匹配
    "Vertiacl_Shading_Top_Left_Movement_Distance": "M_TL",
    "Vertiacl_Shading_Top_Right_Movement_Distance": "M_TR"
}


# =============================================================================
# 第二部分：漂亮标签映射表 (Pretty Labels Dict)
# 用途：仅用于 SHAP 绘图时的 feature_names 参数
# 特点：使用 LaTeX 语法 ($...$) 渲染下标和希腊字母，SVG 导出后在 Figma 中可编辑。
# 注意：Key 必须与上面 safe_rename_dict 的 Value 完全一致。
# =============================================================================
pretty_labels_dict = {
    # 基础几何
    "WWR_s": "$WWR_s$",
    "H_s": "$H_s$",
    "W_s": "$W_s$",
    "SH_s": "$SH_s$",
    "d_A_s": "$d_{A,s}$",
    "d_B_s": "$d_{B,s}$",
    
    "WWR_c": "$WWR_c$",
    "H_c": "$H_c$",
    "W_c": "$W_c$",
    "SH_c": "$SH_c$",
    "d_A_c": "$d_{A,c}$",
    "d_B_c": "$d_{B,c}$",
    
    # 悬挑系统
    "alpha_oh": r"$\alpha_{oh}$",
    "L_oh": "$L_{oh}$",
    "d_mv": "$d_{mv}$",
    
    # 垂直系统
    "N_v": "$N_v$",
    "W_v": "$W_v$",
    "L_v": "$L_v$",
    "M_TL": "$M_{TL}$",
    "M_TR": "$M_{TR}$",
    
    # 框架系统
    "F_BL": "$F_{BL}$",
    "F_BR": "$F_{BR}$",
    "F_TL": "$F_{TL}$",
    "F_TR": "$F_{TR}$"
}
# ================== 数据准备与清理 ==================
targets = data[['out:EUI', 'out:sDA', 'out:sGA', 'out:UDI', 'out:SVF']]
features = data.drop(columns=targets.columns)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(features, targets, test_size=0.2, random_state=42)

def clean_and_rename_columns(df, mapping):
    new_cols = df.columns.str.replace('"', '', regex=False) \
                         .str.replace('in:', '', regex=False) \
                         .str.replace('out:', '', regex=False)
    df.columns = new_cols
    return df.rename(columns=mapping)

# 执行重命名为安全名称
X_train = clean_and_rename_columns(X_train_raw.copy(), safe_rename_dict)
X_test = clean_and_rename_columns(X_test_raw.copy(), safe_rename_dict)

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# 保存数据
csv_filename = Path(file_path).stem
save_dir = Path(file_path).parent / csv_filename
save_dir.mkdir(exist_ok=True)
joblib.dump(X_train, save_dir / "X_train.joblib")
joblib.dump(y_train, save_dir / "y_train.joblib")
joblib.dump(X_test, save_dir / "X_test.joblib")
joblib.dump(y_test, save_dir / "y_test.joblib")

print("数据安全重命名完成，模型训练将使用以下列：", X_train.columns.tolist())

数据安全重命名完成，模型训练将使用以下列： ['WWR_s', 'W_s', 'H_s', 'SH_s', 'd_A_s', 'd_B_s', 'WWR_c', 'H_c', 'W_c', 'SH_c', 'd_A_c', 'd_B_c', 'alpha_oh', 'L_oh', 'd_mv', 'N_v', 'W_v', 'L_v', 'M_TL', 'M_TR']


In [4]:
all_base_models = []  # 存储所有基模型
all_meta_models = []  # 存储所有元模型
all_target_names = [] # 存储目标变量名称

def save_figure(fig, filename):
    # 【关键修改】保存时增加 transparent=True 和可编辑文字支持
    save_path = save_dir / f"{filename}.svg"
    fig.savefig(save_path, format='svg', bbox_inches='tight', transparent=True)
    print(f"矢量图已保存（Figma文字可编辑）: {save_path}")

def save_model(model, filename):
    save_path = save_dir / f"{filename}.joblib"
    joblib.dump(model, save_path)

clean_targets = [target.replace("out:", "") for target in y_train.columns]

In [5]:
# ================== 模型评估函数定义 ==================
# 确保在此单元格或前序单元格中已定义以下函数

def xgb_evaluate(learning_rate, n_estimators, max_depth, subsample, colsample_bytree, X_train, y_train_obj):
    params = {
        'learning_rate': learning_rate,
        'n_estimators': int(n_estimators),
        'max_depth': int(max_depth),
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'objective': 'reg:squarederror',
        'random_state': 42
    }
    model = XGBRegressor(**params)
    scores = cross_val_score(model, X_train, y_train_obj, cv=10, scoring='neg_mean_squared_error')
    return np.mean(scores)

def lgbm_evaluate(learning_rate, n_estimators, max_depth, num_leaves, X_train, y_train_obj):
    params = {
        'learning_rate': learning_rate,
        'n_estimators': int(n_estimators),
        'max_depth': int(max_depth),
        'num_leaves': int(num_leaves),
        'objective': 'regression',
        'random_state': 42,
        'verbose': -1
    }
    model = LGBMRegressor(**params)
    scores = cross_val_score(model, X_train, y_train_obj, cv=10, scoring='neg_mean_squared_error')
    return np.mean(scores)

def rf_evaluate(n_estimators, max_depth, min_samples_split, min_samples_leaf, X_train, y_train_obj):
    params = {
        'n_estimators': int(n_estimators),
        'max_depth': int(max_depth),
        'min_samples_split': int(min_samples_split),
        'min_samples_leaf': int(min_samples_leaf),
        'random_state': 42,
        'n_jobs': -1
    }
    model = RandomForestRegressor(**params)
    scores = cross_val_score(model, X_train, y_train_obj, cv=10, scoring='neg_mean_squared_error')
    return np.mean(scores)

# ================== 绘图环境全局配置 ==================
plt.rcParams['svg.fonttype'] = 'none'        # Figma可编辑文字核心设置
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'   # 使 LaTeX 公式风格与 Times 一致
plt.rcParams['font.size'] = 12

# ================== Stacking 训练与可视化核心循环 ==================

# 确保在 Cell 2 定义了 pretty_labels_dict
# 如果没有定义，请取消下面注释手动定义一次：
# pretty_labels_dict = {col: f"${col}$" for col in X_train.columns} 

for i in range(5):
    target_name = clean_targets[i]
    # 修正此处引号导致的 SyntaxError
    print(f"\n{'='*20} 正在处理目标变量: {target_name} {'='*20}")
    y_train_obj = y_train.iloc[:, i]
    y_test_obj = y_test.iloc[:, i]

    # --- 1. XGBoost 贝叶斯优化 ---
    print(f"正在优化 XGBoost...")
    optimizer_xgb = BayesianOptimization(
        f=lambda learning_rate, n_estimators, max_depth, subsample, colsample_bytree: 
            xgb_evaluate(learning_rate, n_estimators, max_depth, subsample, colsample_bytree, X_train, y_train_obj),
        pbounds={'learning_rate': (0.01, 0.3), 'n_estimators': (50, 500), 'max_depth': (3, 15), 'subsample': (0.5, 1.0), 'colsample_bytree': (0.5, 1.0)},
        random_state=42, verbose=0
    )
    optimizer_xgb.maximize(init_points=5, n_iter=25)
    best_params_xgb = optimizer_xgb.max['params']
    best_params_xgb['n_estimators'] = int(best_params_xgb['n_estimators'])
    best_params_xgb['max_depth'] = int(best_params_xgb['max_depth'])

    # --- 2. LightGBM 贝叶斯优化 ---
    print(f"正在优化 LightGBM...")
    optimizer_lgbm = BayesianOptimization(
        f=lambda learning_rate, n_estimators, max_depth, num_leaves: 
            lgbm_evaluate(learning_rate, n_estimators, max_depth, num_leaves, X_train, y_train_obj),
        pbounds={'learning_rate': (0.01, 0.3), 'n_estimators': (50, 500), 'max_depth': (3, 15), 'num_leaves': (20, 100)},
        random_state=42, verbose=0
    )
    optimizer_lgbm.maximize(init_points=5, n_iter=25)
    best_params_lgbm = optimizer_lgbm.max['params']
    best_params_lgbm['n_estimators'] = int(best_params_lgbm['n_estimators'])
    best_params_lgbm['max_depth'] = int(best_params_lgbm['max_depth'])
    best_params_lgbm['num_leaves'] = int(best_params_lgbm['num_leaves'])

    # --- 3. Random Forest 贝叶斯优化 ---
    print(f"正在优化 Random Forest...")
    optimizer_rf = BayesianOptimization(
        f=lambda n_estimators, max_depth, min_samples_split, min_samples_leaf: 
            rf_evaluate(n_estimators, max_depth, min_samples_split, min_samples_leaf, X_train, y_train_obj),
        pbounds={'n_estimators': (50, 500), 'max_depth': (3, 15), 'min_samples_split': (2, 10), 'min_samples_leaf': (1, 10)},
        random_state=42, verbose=0
    )
    optimizer_rf.maximize(init_points=5, n_iter=25)
    best_params_rf = optimizer_rf.max['params']
    best_params_rf['n_estimators'] = int(best_params_rf['n_estimators'])
    best_params_rf['max_depth'] = int(best_params_rf['max_depth'])
    best_params_rf['min_samples_split'] = int(best_params_rf['min_samples_split'])
    best_params_rf['min_samples_leaf'] = int(best_params_rf['min_samples_leaf'])

    # --- 4. 构建 Stacking 模型 ---
    base_models = [
        ('xgb', XGBRegressor(**best_params_xgb, random_state=42)),
        ('lgbm', LGBMRegressor(**best_params_lgbm, random_state=42, verbose=-1)),
        ('rf', RandomForestRegressor(**best_params_rf, random_state=42))
    ]

    train_predictions = []
    test_predictions = []
    for name, model in base_models:
        # 使用 OOF (Out-Of-Fold) 预测生成元特征
        train_pred = cross_val_predict(model, X_train, y_train_obj, cv=10)
        train_predictions.append(train_pred)
        # 全量拟合用于测试集预测
        model.fit(X_train, y_train_obj)
        test_predictions.append(model.predict(X_test))

    # 训练元模型
    meta_model = LinearRegression()
    meta_model.fit(np.column_stack(train_predictions), y_train_obj)
    
    # 记录到全局列表
    all_base_models.append(base_models)
    all_meta_models.append(meta_model)
    all_target_names.append(target_name)

    # --- 5. SHAP 分析与绘图 (核心修正) ---
    print(f"正在为 {target_name} 计算 SHAP 并导出矢量图...")
    xgb_trained_model = base_models[0][1] # 取出拟合好的 XGBoost
    
    # 获取用于绘图的 LaTeX 标签列表
    current_pretty_names = [pretty_labels_dict.get(col, col) for col in X_test.columns]
    
    try:
        explainer = shap.TreeExplainer(xgb_trained_model)
        shap_values_obj = explainer(X_test)
        shap_v = shap_values_obj.values if hasattr(shap_values_obj, "values") else shap_values_obj
    except Exception as e:
        print(f"TreeExplainer 异常，尝试 Permutation 模式: {e}")
        explainer = shap.Explainer(xgb_trained_model.predict, X_train.iloc[:100, :])
        shap_v = explainer(X_test).values

    # 绘制 Bar Plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_v, X_test, plot_type="bar", feature_names=current_pretty_names, show=False)
    plt.title(f'SHAP Feature Importance - {target_name}', fontdict={'weight':'bold', 'size':14})
    save_figure(plt.gcf(), f"SHAP_bar_{target_name}")
    plt.close()

    # 绘制 Dot Plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_v, X_test, plot_type="dot", feature_names=current_pretty_names, show=False)
    plt.title(f"SHAP Impact Distribution - {target_name}", fontdict={'weight':'bold', 'size':14})
    save_figure(plt.gcf(), f"SHAP_dot_{target_name}")
    plt.close()

    # --- 6. 保存模型文件 ---
    save_model(base_models[0][1], f"xgb_model_{target_name}")
    save_model(base_models[1][1], f"lgbm_model_{target_name}")
    save_model(base_models[2][1], f"rf_model_{target_name}")
    save_model(meta_model, f"meta_model_{target_name}")

print("\n✅ 所有模型训练与可视化生成任务已成功完成，图表可直接拖入 Figma 编辑。")


==================== 正在处理目标变量: EUI ====================
正在优化 XGBoost...
正在优化 LightGBM...
正在优化 Random Forest...
正在为 EUI 计算 SHAP 并导出矢量图...
TreeExplainer 异常，尝试 Permutation 模式: could not convert string to float: '[8.245179E1]'


PermutationExplainer explainer: 401it [00:18, 13.47it/s]                         
C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_54384\1226594048.py:150: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_v, X_test, plot_type="bar", feature_names=current_pretty_names, show=False)


矢量图已保存（Figma文字可编辑）: C:\Users\GIGABYTE\Desktop\Test\Analysis\新（可用）\CSV\0312_2000_North0°_Overhang+Vertical(1)_replaced\SHAP_bar_EUI.svg


C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_54384\1226594048.py:157: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_v, X_test, plot_type="dot", feature_names=current_pretty_names, show=False)


矢量图已保存（Figma文字可编辑）: C:\Users\GIGABYTE\Desktop\Test\Analysis\新（可用）\CSV\0312_2000_North0°_Overhang+Vertical(1)_replaced\SHAP_dot_EUI.svg

==================== 正在处理目标变量: sDA ====================
正在优化 XGBoost...
正在优化 LightGBM...
正在优化 Random Forest...
正在为 sDA 计算 SHAP 并导出矢量图...
TreeExplainer 异常，尝试 Permutation 模式: could not convert string to float: '[8.9050735E1]'


...

矢量图已保存（Figma文字可编辑）: C:\Users\GIGABYTE\Desktop\Test\Analysis\新（可用）\CSV\0312_2000_North0°_Overhang+Vertical(1)_replaced\SHAP_dot_UDI.svg

==================== 正在处理目标变量: SVF ====================
正在优化 XGBoost...
正在优化 LightGBM...
正在优化 Random Forest...
正在为 SVF 计算 SHAP 并导出矢量图...
TreeExplainer 异常，尝试 Permutation 模式: could not convert string to float: '[2.8213562E1]'


PermutationExplainer explainer: 401it [00:11,  5.61it/s]                         
C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_54384\1226594048.py:150: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_v, X_test, plot_type="bar", feature_names=current_pretty_names, show=False)


矢量图已保存（Figma文字可编辑）: C:\Users\GIGABYTE\Desktop\Test\Analysis\新（可用）\CSV\0312_2000_North0°_Overhang+Vertical(1)_replaced\SHAP_bar_SVF.svg


C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_54384\1226594048.py:157: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_v, X_test, plot_type="dot", feature_names=current_pretty_names, show=False)


矢量图已保存（Figma文字可编辑）: C:\Users\GIGABYTE\Desktop\Test\Analysis\新（可用）\CSV\0312_2000_North0°_Overhang+Vertical(1)_replaced\SHAP_dot_SVF.svg

✅ 所有模型训练与可视化生成任务已成功完成，图表可直接拖入 Figma 编辑。


In [6]:
# ================== 批量评估所有已训练模型的 R² 与 RMSE ==================
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error

# 输入：模型所在的父目录（即包含所有子文件夹的目录）
parent_dir = Path(r"C:\Users\GIGABYTE\Desktop\Test\Analysis\新（可用）\CSV")

target_names = ["EUI", "sDA", "sGA", "UDI", "SVF"]
model_types = ["xgb", "lgbm", "rf"]

results = []

for folder in sorted(parent_dir.iterdir()):
    if not folder.is_dir():
        continue
    
    # 检查该文件夹是否包含必要的测试数据文件
    x_test_path = folder / "X_test.joblib"
    y_test_path = folder / "y_test.joblib"
    if not x_test_path.exists() or not y_test_path.exists():
        continue
    
    folder_name = folder.name
    print(f"正在处理: {folder_name}")
    
    X_test_eval = joblib.load(x_test_path)
    y_test_eval = joblib.load(y_test_path)
    
    for target in target_names:
        # y_test_eval 可能是 DataFrame（多列）或 Series
        if isinstance(y_test_eval, pd.DataFrame):
            # 找到包含该目标名称的列
            matched_cols = [c for c in y_test_eval.columns if target in c]
            if not matched_cols:
                continue
            y_true = y_test_eval[matched_cols[0]].values
        else:
            continue
        
        # 加载各基模型并评估
        base_preds = []
        for mtype in model_types:
            model_path = folder / f"{mtype}_model_{target}.joblib"
            if not model_path.exists():
                break
            model = joblib.load(model_path)
            y_pred = model.predict(X_test_eval)
            r2 = r2_score(y_true, y_pred)
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            
            model_label = {"xgb": "XGBoost", "lgbm": "LightGBM", "rf": "RandomForest"}[mtype]
            results.append({
                "Folder": folder_name,
                "Target": target,
                "Model": model_label,
                "R²": round(r2, 6),
                "RMSE": round(rmse, 6)
            })
            base_preds.append(y_pred)
        
        # Stacking 评估：用三个基模型的预测作为元特征，输入元模型
        meta_path = folder / f"meta_model_{target}.joblib"
        if len(base_preds) == 3 and meta_path.exists():
            meta_model = joblib.load(meta_path)
            meta_input = np.column_stack(base_preds)
            y_pred_stack = meta_model.predict(meta_input)
            r2_stack = r2_score(y_true, y_pred_stack)
            rmse_stack = np.sqrt(mean_squared_error(y_true, y_pred_stack))
            results.append({
                "Folder": folder_name,
                "Target": target,
                "Model": "Stacking",
                "R²": round(r2_stack, 6),
                "RMSE": round(rmse_stack, 6)
            })

# 整合为 DataFrame 并保存
df_results = pd.DataFrame(results)
output_csv = parent_dir / "模型评估汇总_R2_RMSE.csv"
df_results.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"\n✅ 评估完成！共 {len(df_results)} 条记录")
print(f"结果已保存至: {output_csv}\n")
df_results

正在处理: 0304_1500_South0°_Frame(1)
正在处理: 0304_2000_South0°_Overhang(1)
正在处理: 0304_800_South0°(1)
正在处理: 0305_800_North0°(1)
正在处理: 0306_1500_North0°_Frame(1)
正在处理: 0306_2000_North0°_Overhang(1)
正在处理: 0307_2000_South0°_Vertical(1)
正在处理: 0310_2000_North0°_Vertical(1)
正在处理: 0311_2000_South0°_Overhang+Vertical(1)
正在处理: 0312_2000_North0°_Overhang+Vertical(1)
正在处理: 0312_2000_North0°_Overhang+Vertical(1)_replaced

✅ 评估完成！共 220 条记录
结果已保存至: C:\Users\GIGABYTE\Desktop\Test\Analysis\新（可用）\CSV\模型评估汇总_R2_RMSE.csv



,Folder,Target,Model,R²,RMSE
0,0304_1500_South0°_Frame(1),EUI,XGBoost,0.989562,0.134788
1,0304_1500_South0°_Frame(1),EUI,LightGBM,0.992703,0.112694
2,0304_1500_South0°_Frame(1),EUI,RandomForest,0.970727,0.225722
3,0304_1500_South0°_Frame(1),EUI,Stacking,0.993762,0.104201
4,0304_1500_South0°_Frame(1),sDA,XGBoost,0.974666,0.010741
...,...,...,...,...,...
215,0312_2000_North0°_Overhang+Vertical(1)_replaced,UDI,Stacking,0.945183,1.381280
216,0312_2000_North0°_Overhang+Vertical(1)_replaced,SVF,XGBoost,0.967736,1.546217
217,0312_2000_North0°_Overhang+Vertical(1)_replaced,SVF,LightGBM,0.971645,1.449522
218,0312_2000_North0°_Overhang+Vertical(1)_replaced,SVF,RandomForest,0.936728,2.165276
